In [1]:
# I was looking at the recent rally in the SP500 index over the last week, an enormous move of 13.21% from low to high. 
# This made me wonder, when we see a 10%+ move in the index from low to high (or low to close better?), what comes next over 1M, 3M, 6M, 12M, 24 months?

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

In [51]:
ticker = 'SPY'
date = '1900-01-01'

data = yf.download(ticker, date, interval='1wk', multi_level_index=False, auto_adjust=True, progress=False)

data = data.drop(columns=['Volume'])
data.tail(10)

,Close,High,Low,Open
Date,,,,
2026-02-09,679.893188,695.241287,675.674729,687.542281
2026-02-16,687.552307,688.180596,673.939520,678.287631
2026-02-23,684.121643,691.790701,678.147967,685.956658
2026-03-02,670.548706,686.744465,667.836083,676.851500
2026-03-09,660.486206,681.498828,659.558746,664.575076
2026-03-16,646.803589,672.603126,642.964038,666.559633
2026-03-23,634.090027,662.619995,633.109985,658.070007
2026-03-30,655.830017,658.520020,629.280029,640.109985
2026-04-06,679.460022,682.030029,651.059998,655.859985


In [52]:
# add a column tracking percentage gain of current close vs close of two weeks earlier (current week, 1 week ago, 2 weeks ago for three week rolling), probably best to use shift(2). 
# then, add another column which tracks signals. 1 if the 3 week rolling gain is over 10% (or specified value) and 0 if not. 
# might be a good idea to add a buffer, so that you cannot have a signal twice in a few weeks or a month. So track just the first instance of a signal. 

data['3wk_ret'] = ((data['Close'] / data['Close'].shift(2)) - 1) * 100 # 6 outputs at 10%
#data['3wk_ret'] = ((data['High'] / data['Low'].shift(2)) - 1) * 100 # 64 outputs at 10%
data

,Close,High,Low,Open,3wk_ret
Date,,,,,
1993-01-25,24.175379,24.192573,24.072212,24.192573,NaN
1993-02-01,24.742802,24.811579,24.192580,24.192580,NaN
1993-02-08,24.536467,24.828772,24.502078,24.742800,1.493618
1993-02-15,23.969051,24.467690,23.556385,24.467690,-3.127173
1993-02-22,24.433292,24.450487,23.917459,24.037820,-0.420493
...,...,...,...,...,...
2026-03-16,646.803589,672.603126,642.964038,666.559633,-3.541147
2026-03-23,634.090027,662.619995,633.109985,658.070007,-3.996477
2026-03-30,655.830017,658.520020,629.280029,640.109985,1.395544


In [53]:
signal = 7.5

data['signal'] = np.where(data['3wk_ret'] >=signal, 1, 0)
data

,Close,High,Low,Open,3wk_ret,signal
Date,,,,,,
1993-01-25,24.175379,24.192573,24.072212,24.192573,NaN,0
1993-02-01,24.742802,24.811579,24.192580,24.192580,NaN,0
1993-02-08,24.536467,24.828772,24.502078,24.742800,1.493618,0
1993-02-15,23.969051,24.467690,23.556385,24.467690,-3.127173,0
1993-02-22,24.433292,24.450487,23.917459,24.037820,-0.420493,0
...,...,...,...,...,...,...
2026-03-16,646.803589,672.603126,642.964038,666.559633,-3.541147,0
2026-03-23,634.090027,662.619995,633.109985,658.070007,-3.996477,0
2026-03-30,655.830017,658.520020,629.280029,640.109985,1.395544,0


In [54]:
data.loc[data['signal'] == 1]


,Close,High,Low,Open,3wk_ret,signal
Date,,,,,,
1997-05-05,50.116348,50.684991,49.244430,49.528751,7.962423,1
1998-10-19,66.187630,67.397553,65.355203,65.471355,8.436384,1
1999-10-25,85.856155,86.287003,80.098466,81.038497,9.709737,1
2000-03-20,96.717018,98.094753,91.009258,92.505084,9.868125,1
2001-04-16,79.249443,80.204256,74.418094,75.296520,9.885303,1
2001-10-01,68.707291,69.822189,65.888003,66.573602,10.641733,1
2002-10-14,57.611584,58.040553,53.971864,54.075853,9.702975,1
2003-03-17,58.567432,58.704592,54.354655,54.511408,7.621181,1
2008-11-03,68.086700,73.164549,65.330152,70.204888,7.835448,1


In [ ]:
# next step is to add a signal_buffer column so that we only record the first instance of a signal. We don't want the same period flagging twice. 
data['signal_buffer'] = np.where((data['signal'] == 1) 
                                 & (data['signal'].shift(1) != 1)
                                 & (data['signal'].shift(2) != 1)
                                 & (data['signal'].shift(3) != 1), 1, 0)

data.loc[data['signal'] == 1]

# probably cleaner with a 4wk rolling sum implementation in the np.where condition.  
# data['rolling'] = data['signal'].rolling(4).sum().shift(1)

,Close,High,Low,Open,3wk_ret,signal,signal_buffer
Date,,,,,,,
1997-05-05,50.116348,50.684991,49.244430,49.528751,7.962423,1,1
1998-10-19,66.187630,67.397553,65.355203,65.471355,8.436384,1,1
1999-10-25,85.856155,86.287003,80.098466,81.038497,9.709737,1,1
2000-03-20,96.717018,98.094753,91.009258,92.505084,9.868125,1,1
2001-04-16,79.249443,80.204256,74.418094,75.296520,9.885303,1,1
2001-10-01,68.707291,69.822189,65.888003,66.573602,10.641733,1,1
2002-10-14,57.611584,58.040553,53.971864,54.075853,9.702975,1,1
2003-03-17,58.567432,58.704592,54.354655,54.511408,7.621181,1,1
2008-11-03,68.086700,73.164549,65.330152,70.204888,7.835448,1,1
